In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
# 0 = all messages are logged (default behavior)
# 1 = INFO messages are not printed
# 2 = INFO and WARNING messages are not printed
# 3 = INFO, WARNING, and ERROR messages are not printed

import datetime

import IPython
import IPython.display
import matplotlib as mpl
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
#from google.colab import drive
#drive.mount('/content/gdrive')

mpl.rcParams['figure.figsize'] = (8, 6)
mpl.rcParams['axes.grid'] = False

import warnings
#https://stackoverflow.com/questions/15777951/how-to-suppress-pandas-future-warning
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

import logging
tf.get_logger().setLevel(logging.ERROR)

# https://stackoverflow.com/questions/65697623/tensorflow-warning-found-untraced-functions-such-as-lstm-cell-6-layer-call-and
import absl.logging

In [ ]:

import argparse
import numpy as np
import pandas as pd
import tensorflow as tf

import math
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from keras.models import load_model

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Fri Aug 25 16:23:14 2023       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.105.17   Driver Version: 525.105.17   CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla V100-SXM2...  Off  | 00000000:00:04.0 Off |                    0 |
| N/A   33C    P0    25W / 300W |      0MiB / 16384MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')
%cd /content/drive/My Drive/Colab Notebooks/10W7S_single_step/TrialTwo/

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
[Errno 2] No such file or directory: '/content/drive/My Drive/Colab Notebooks/TrialTwo/10W7S_single_step/'
/content


In [ ]:
%run "Data_Helpers.ipynb"
cur_df = dfs["Station2"]
%cd /content/drive/My Drive/Colab Notebooks/10W7S_single_step/TrialTwo/

Exception: ignored

In [ ]:
# MODEL DEFS
msd_model = tf.keras.Sequential([
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units=64, activation='tanh'),
    tf.keras.layers.Dense(units=16, activation='relu'),
    tf.keras.layers.Dense(units=1),
    tf.keras.layers.Reshape([1, -1]),
])

cnn_model = tf.keras.Sequential([
    tf.keras.layers.Conv1D(filters=32,
                           kernel_size=(24*10,),
                           activation='relu'),
    tf.keras.layers.Dense(units=64, activation='relu'),
    tf.keras.layers.Dense(units=1),
])

lstm_model = tf.keras.models.Sequential([
    tf.keras.layers.LSTM(64, return_sequences=True),
    tf.keras.layers.LSTM(64, return_sequences=False),
    tf.keras.layers.Dense(units=64, activation='relu'),
    tf.keras.layers.Dense(units=16, activation='relu'),
    tf.keras.layers.Dense(units=1, activation='tanh')
])

In [ ]:
class FeedBack(tf.keras.Model):
    def __init__(self, units, out_steps, num_features):
        super().__init__()
        self.out_steps = out_steps
        self.units = units
        self.lstm_cell = tf.keras.layers.LSTMCell(units)
        self.lstm_rnn = tf.keras.layers.RNN(self.lstm_cell, return_state=True)
        self.dense = tf.keras.layers.Dense(num_features)

    def call(self, inputs, training=None):
        # Use a TensorArray to capture dynamically unrolled outputs.
        predictions = []

        # Initialize the LSTM state.
        x, *state = self.lstm_rnn(inputs)

        # This is the prediction for the first time step
        prediction = self.dense(x)

        # Insert the first prediction.
        predictions.append(prediction)

        # Run the rest of the prediction steps.
        for n in range(1, self.out_steps):
            # Use the last prediction as input.
            x = prediction
            # Execute one lstm step.
            x, state = self.lstm_cell(x, states=state,
                              training=training)
            # Convert the lstm output to a prediction.
            prediction = self.dense(x)
            # Add the prediction to the output.
            predictions.append(prediction)

        # predictions.shape => (time, batch, features)
        predictions = tf.stack(predictions)
        # predictions.shape => (batch, time, features)
        predictions = tf.transpose(predictions, [1, 0, 2])
        return predictions



feedback_model = FeedBack(units=64, out_steps=1, num_features=1)

In [ ]:
model_list = [msd_model, cnn_model, lstm_model,feedback_model]
model_names = ["MSD","CNN","RNN","AR RNN"] #
levels = ["SWC_5", "SWC_10","SWC_20", "SWC_50"]
data_levels = ["NoMet", "Met"]

In [ ]:
loss_by_epoch = {}
val_performance = {}
performance = {}

In [ ]:
import csv
TARGET_COL = ""
TRAIN_SPLIT = 0.7
VAL_SPLIT = 0.2
WINDOW_SIZE = 24*10
SHIFT_AMT = 24*7
BATCH_SIZE = 64
MAX_EPOCHS = 50

def run_all_models():
  for data_level in data_levels:
    if data_level == "NoMet":
      df = cur_df[["SWC_5","SWC_10","SWC_20","SWC_50","Day sin", "Day cos","Year sin", "Year cos"]]
    else:
      df = cur_df
    for level in levels:
      TARGET_COL = level
      print(TARGET_COL)
      TRAIN_SPLIT = 0.7
      VAL_SPLIT = 0.2
      WINDOW_SIZE = 24*10
      SHIFT_AMT = 24*7
      BATCH_SIZE = 64
      MAX_EPOCHS = 50
      %run "Window_Helpers.ipynb"

      X_train, y_train, X_val, y_val, X_test, y_test = preprocess_data(df, TARGET_COL)
      train_dataset, train_steps = generate_batches(X_train, y_train, batch_size=BATCH_SIZE)
      val_dataset, val_steps = generate_batches(X_val, y_val, batch_size=BATCH_SIZE)
      test_dataset, test_steps = generate_batches(X_test, y_test, batch_size=BATCH_SIZE)


      %run "Model_Runners.ipynb"
      for i in range(0,len(model_list)):
        model_name = f"{model_names[i]}_{level}_{data_level}"
        print(model_name, '\n')
        compile_and_fit(model_list[i], train_dataset, train_steps, val_dataset, val_steps,test_dataset, test_steps, batch_size=BATCH_SIZE, model_name=model_name, max_epochs=MAX_EPOCHS, patience=50)
        if model_names[i] == "RNN":
          pred = model_list[i].predict(test_dataset, batch_size=BATCH_SIZE, steps=test_steps)[:, 0]
        else:
          pred = model_list[i].predict(test_dataset, batch_size=BATCH_SIZE, steps=test_steps)[:, 0, 0]
        row = [model_names[i],data_level, TARGET_COL, WINDOW_SIZE, SHIFT_AMT, MAX_EPOCHS, performance[model_name][1], val_performance[model_name][1], loss_by_epoch[model_name], y_test, pred]
        with open('results.csv', 'a') as file:
          writer = csv.writer(file)
          writer.writerow(row)
          file.close()





In [ ]:
run_all_models()


SWC_10
Index(['SWC_5', 'SWC_10', 'SWC_20', 'SWC_50', 'Day sin', 'Day cos', 'Year sin',
       'Year cos'],
      dtype='object')
AR RNN_SWC_10_NoMet 

Epoch 1/50
595/595 [==============================] - ETA: 0s - loss: 0.0406 - mean_absolute_error: 0.1571 - mean_squared_error: 0.0406

595/595 [==============================] - 72s 110ms/step - loss: 0.0406 - mean_absolute_error: 0.1571 - mean_squared_error: 0.0406 - val_loss: 0.0258 - val_mean_absolute_error: 0.1308 - val_mean_squared_error: 0.0258
Epoch 2/50
595/595 [==============================] - 63s 107ms/step - loss: 0.0341 - mean_absolute_error: 0.1426 - mean_squared_error: 0.0341 - val_loss: 0.0259 - val_mean_absolute_error: 0.1307 - val_mean_squared_error: 0.0259
Epoch 3/50
595/595 [==============================] - 63s 106ms/step - loss: 0.0336 - mean_absolute_error: 0.1415 - mean_squared_error: 0.0336 - val_loss: 0.0261 - val_mean_absolute_error: 0.1313 - val_mean_squared_error: 0.0261
Epoch 4/50
595/595 [==============================] - 63s 106ms/step - loss: 0.0335 - mean_absolute_error: 0.1412 - mean_squared_error: 0.0335 - val_loss: 0.0262 - val_mean_absolute_error: 0.1316 - val_mean_squared_error: 0.0262
Epoch 5/50
595/595 [==============================] - 63s 106ms/step - loss: 0.0333 - mean_abso

595/595 [==============================] - 64s 107ms/step - loss: 0.0321 - mean_absolute_error: 0.1379 - mean_squared_error: 0.0321 - val_loss: 0.0258 - val_mean_absolute_error: 0.1290 - val_mean_squared_error: 0.0258
Epoch 20/50
595/595 [==============================] - ETA: 0s - loss: 0.0321 - mean_absolute_error: 0.1378 - mean_squared_error: 0.0321

595/595 [==============================] - 65s 109ms/step - loss: 0.0321 - mean_absolute_error: 0.1378 - mean_squared_error: 0.0321 - val_loss: 0.0257 - val_mean_absolute_error: 0.1286 - val_mean_squared_error: 0.0257
Epoch 21/50
595/595 [==============================] - ETA: 0s - loss: 0.0321 - mean_absolute_error: 0.1376 - mean_squared_error: 0.0321

595/595 [==============================] - 64s 107ms/step - loss: 0.0321 - mean_absolute_error: 0.1376 - mean_squared_error: 0.0321 - val_loss: 0.0257 - val_mean_absolute_error: 0.1284 - val_mean_squared_error: 0.0257
Epoch 22/50
595/595 [==============================] - ETA: 0s - loss: 0.0321 - mean_absolute_error: 0.1376 - mean_squared_error: 0.0321

...

595/595 [==============================] - 65s 109ms/step - loss: 0.0315 - mean_absolute_error: 0.1360 - mean_squared_error: 0.0315 - val_loss: 0.0251 - val_mean_absolute_error: 0.1250 - val_mean_squared_error: 0.0251
Epoch 33/50
595/595 [==============================] - ETA: 0s - loss: 0.0315 - mean_absolute_error: 0.1360 - mean_squared_error: 0.0315

595/595 [==============================] - 64s 107ms/step - loss: 0.0315 - mean_absolute_error: 0.1360 - mean_squared_error: 0.0315 - val_loss: 0.0250 - val_mean_absolute_error: 0.1247 - val_mean_squared_error: 0.0250
Epoch 34/50
595/595 [==============================] - ETA: 0s - loss: 0.0314 - mean_absolute_error: 0.1356 - mean_squared_error: 0.0314

595/595 [==============================] - 65s 109ms/step - loss: 0.0314 - mean_absolute_error: 0.1356 - mean_squared_error: 0.0314 - val_loss: 0.0250 - val_mean_absolute_error: 0.1243 - val_mean_squared_error: 0.0250
Epoch 35/50
595/595 [==============================] - 63s 106ms/step - loss: 0.0318 - mean_absolute_error: 0.1366 - mean_squared_error: 0.0318 - val_loss: 0.0251 - val_mean_absolute_error: 0.1240 - val_mean_squared_error: 0.0251
Epoch 36/50
595/595 [==============================] - 63s 107ms/step - loss: 0.0313 - mean_absolute_error: 0.1357 - mean_squared_error: 0.0313 - val_loss: 0.0251 - val_mean_absolute_error: 0.1239 - val_mean_squared_error: 0.0251
Epoch 37/50
595/595 [==============================] - 63s 106ms/step - loss: 0.0313 - mean_absolute_error: 0.1356 - mean_squared_error: 0.0313 - val_loss: 0.0251 - val_mean_absolute_error: 0.1238 - val_mean_squared_error: 0.0251
Epoch 38/50
595/595 [==============================] - 64s 107ms/step - loss: 0.0312 - mean_

595/595 [==============================] - 65s 109ms/step - loss: 0.0311 - mean_absolute_error: 0.1347 - mean_squared_error: 0.0311 - val_loss: 0.0249 - val_mean_absolute_error: 0.1232 - val_mean_squared_error: 0.0249
Epoch 40/50
595/595 [==============================] - ETA: 0s - loss: 0.0311 - mean_absolute_error: 0.1347 - mean_squared_error: 0.0311

595/595 [==============================] - 65s 109ms/step - loss: 0.0311 - mean_absolute_error: 0.1347 - mean_squared_error: 0.0311 - val_loss: 0.0248 - val_mean_absolute_error: 0.1209 - val_mean_squared_error: 0.0248
Epoch 41/50
595/595 [==============================] - 62s 105ms/step - loss: 0.0312 - mean_absolute_error: 0.1355 - mean_squared_error: 0.0312 - val_loss: 0.0251 - val_mean_absolute_error: 0.1227 - val_mean_squared_error: 0.0251
Epoch 42/50
595/595 [==============================] - 65s 109ms/step - loss: 0.0308 - mean_absolute_error: 0.1341 - mean_squared_error: 0.0308 - val_loss: 0.0256 - val_mean_absolute_error: 0.1224 - val_mean_squared_error: 0.0256
Epoch 43/50
595/595 [==============================] - 63s 106ms/step - loss: 0.0311 - mean_absolute_error: 0.1359 - mean_squared_error: 0.0311 - val_loss: 0.0250 - val_mean_absolute_error: 0.1216 - val_mean_squared_error: 0.0250
Epoch 44/50
595/595 [==============================] - 63s 106ms/step - loss: 0.0306 - mean_

In [ ]:
#MAKE NEW RESULTS CSV
'''''
fields = ["MODEL_NAME","DATA", "TARGET COL","WINDOW_SIZE"," SHIFT_AMT", "MAX_EPOCHS","MAE", "VAL_MAE","LOSS_BY_EPOCH", "Lables", "Predictions"]

with open('results.csv', 'w') as file:
    writer = csv.writer(file)
    writer.writerow(fields)
    file.close()

'''''


'\'\'\nfields = ["MODEL_NAME","TARGET COL","WINDOW_SIZE"," SHIFT_AMT", "MAX_EPOCHS","MAE", "VAL_MAE","LOSS_BY_EPOCH", "Lables", "Predictions"]\n\nwith open(\'results.csv\', \'w\') as file:\n    writer = csv.writer(file)\n    writer.writerow(fields)\n    file.close()\n\n'